## Reference mapping using the CellXGene atlas - BE1

We have previously built the index for all the cells in normal or cancer samples, over 33 million cells in total. You can find the code to build the index at build_atlas_index_faiss.py. We applied careful tuning to eventually well balance between the accuracy and efficiency. Now the actual building process takes less than 3 minutes and we choose to use only 16 bytes to store the vector per cell, which leads to 808 MB for the whole index of all the millions of cells. Please download the faiss index folder from https://drive.google.com/drive/folders/1q14U50SNg5LMjlZ9KH-n-YsGRi8zkCbe?usp=sharing.

Faiss is required to use the index. Please install it by following the instructions at https://github.com/facebookresearch/faiss/wiki/Installing-Faiss

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import mode
import scanpy as sc
import sklearn
import warnings

sys.path.insert(0, "../")
import scgpt as scg

# extra dependency for similarity search
try:
    import faiss

    faiss_imported = True
except ImportError:
    faiss_imported = False
    print(
        "faiss not installed! We highly recommend installing it for fast similarity search."
    )
    print("To install it, see https://github.com/facebookresearch/faiss/wiki/Installing-Faiss")

warnings.filterwarnings("ignore", category=ResourceWarning)

/software/envs/micromamba/envs/nw-scGPTv2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Read the adata concatenated with the embeddings and dividing in ref and test

In [2]:
adata_concat = sc.read_h5ad('/projects/shared/intronic_bam/datasets/anndata/be1_scGPT_embeddings.h5ad')

In [4]:
adata_concat

AnnData object with n_obs × n_vars = 29128 × 24288
    obs: 'Sample', 'Barcode', 'sum', 'detected', 'subsets_Mito_sum', 'subsets_Mito_detected', 'subsets_Mito_percent', 'total', 'discard', 'is_train', 'dataset', 'is_ref', 'Sample_masked', 'predicted_cell_type'
    var: 'ID', 'Symbol', 'Type', 'gene_names', 'id_in_vocab'
    uns: 'Sample_masked_colors', 'is_ref_colors', 'neighbors', 'umap'
    obsm: 'X_scGPT', 'X_umap'
    layers: 'counts'
    obsp: 'connectivities', 'distances'

In [5]:
adata_concat.obs["is_ref"].value_counts()

is_ref
Reference    20397
Query         8731
Name: count, dtype: int64

In [6]:
ref_embd = adata_concat[adata_concat.obs["is_ref"] == "Reference"].copy()
test_embd = adata_concat[adata_concat.obs["is_ref"] == "Query"].copy()

In [7]:
from build_atlas_index_faiss import load_index, vote

In [ ]:
faiss_index_folder = "/projects/shared/intronic_bam/scGPT_supplements/data/cellxgene_faiss/"
use_gpu = faiss.get_num_gpus() > 0
index, meta_labels = load_index(
    index_dir=faiss_index_folder,
    use_config_file=False,
    use_gpu=use_gpu,
)
print(f"Loaded index with {index.ntotal} cells")

In [ ]:
use_gpu

In [1]:
%%time
k = 50
# test with the first 100 cells
distances, idx = index.search(test_embd, k)



CPU times: user 3 μs, sys: 1 μs, total: 4 μs
Wall time: 5.96 μs


NameError: name 'index' is not defined

The search runs super fast, especially on GPU. Here the similarity search for 4,000 query cells within the whole reference of millions should take around 7 second on CPU and 0.1 second on GPU.

In [ ]:
predict_labels = meta_labels[idx]
# from scipy.stats import mode
from tqdm import tqdm

voting = []
for preds in tqdm(predict_labels):
    voting.append(vote(preds, return_prob=False)[0])
voting = np.array(voting)

In [ ]:
from build_atlas_index_faiss import compute_category_proportion
compute_category_proportion(meta_labels)